# CRE follow-up notebook — 870-gene bend-point branch, GO, and FF comparison

This notebook recreates the useful follow-up structure from the older `ipsi_vs_contra_in_ff` notebook, but centers the newer `ipsi_vs_contra_in_cre` branch. The goal is to keep all CRE-side findings in one place: bend-point logic, anchor genes, direction-split GO, ShinyGO-style views, and the comparison back to the WT/FF branch.

The key interpretation is that this is **not a separate hidden dataset or a separate analysis path**. The 870-gene CRE/cKO branch comes from the same 20-sample DRG model framework and the same ordered adjusted-p-value bend-point method used for the 709-gene FF/WT branch.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 180)

MOUSE_ROOT = Path('/Users/pitergarcia/DataScience/Semester5/BIOL550/group_project/mouse_new')
DERIVED_DIR = MOUSE_ROOT / 'preparation' / 'DE-derived_analysis'
FAMILY_DIR = DERIVED_DIR / 'family_structure'
COMPARE_DIR = DERIVED_DIR / 'ff_cre_branch_comparison'
FF_DIR = DERIVED_DIR / 'ipsi_vs_contra_in_ff'
CRE_DIR = DERIVED_DIR / 'ipsi_vs_contra_in_cre'
CRE_SHARED_DIR = CRE_DIR / 'shinygo_style_shared'

for label, path in {
    'DERIVED_DIR': DERIVED_DIR,
    'FF_DIR': FF_DIR,
    'CRE_DIR': CRE_DIR,
    'CRE_SHARED_DIR': CRE_SHARED_DIR,
    'COMPARE_DIR': COMPARE_DIR,
}.items():
    print(f'{label}: {path} ->', 'OK' if path.exists() else 'MISSING')


## Why this notebook exists

The old follow-up work showed that the FF/WT side-specific branch was the cleanest pathway-level story. The newer branch asks the same question in the CRE/cKO background: if we compare ipsilateral vs contralateral DRG in cKO samples, do we recover the same injury-side signal, and does the cKO background add anything beyond the shared injury-response core?

For paper writing, the branch should be framed as a **supporting extension**: FF/WT remains the clearest pathway showcase, while CRE/cKO demonstrates that the side-specific injury response remains strong and expands under the cKO background.


## PCA-first reminder

Before reading the CRE branch as biology, keep the same PCA-first logic used in the earlier reports. Side-specific structure was the strongest sample-level signal, so both side-specific branches deserve follow-up. Genotype remains biologically relevant, but it is secondary to the injury-side contrast in the global sample structure.


In [ ]:
pca_path = FAMILY_DIR / 'pca_side_genotype_annotated.png'
collision_path = FAMILY_DIR / 'pca_ff_cre_collision_pairs.tsv'

if pca_path.exists():
    display(Image(filename=str(pca_path), width=950))
else:
    display(Markdown(f'PCA image not found: `{pca_path}`'))

if collision_path.exists():
    display(pd.read_csv(collision_path, sep='	').head(10))
else:
    display(Markdown(f'Collision-pair table not found: `{collision_path}`'))


## Bend-point checkpoint — `ipsi_vs_contra_in_cre`

This section mirrors the FF bend-point checkpoint. The important point is method consistency: the CRE/cKO branch uses the same ordered adjusted-p-value bend-point rule, so the 870-gene set is a normal consequence of applying the same narrowing rule to a second strong side-specific contrast.


In [ ]:
for image_name in ['before_after_selection_comparison.png', 'ordered_pvalue_and_cumulative_curve.png', 'volcano_with_counts.png']:
    image_path = CRE_DIR / image_name
    if image_path.exists():
        display(Image(filename=str(image_path), width=1000))
    else:
        display(Markdown(f'Missing image: `{image_path}`'))

summary_path = CRE_DIR / 'bendpoint_summary.tsv'
if summary_path.exists():
    bend_summary = pd.read_csv(summary_path, sep='	')
    display(bend_summary)
else:
    display(Markdown(f'Missing bend-point summary: `{summary_path}`'))


## CRE branch size and direction summary

This reproduces the old branch-count logic for CRE/cKO. It separates the full significant set from the narrowed bend-point set and records how many selected genes are upregulated vs downregulated.


In [ ]:
full_path = MOUSE_ROOT / 'preparation' / 'DE-family_drg_novaseqx' / 'tables' / 'ipsi_vs_contra_in_cre_significant.tsv'
selected_path = CRE_DIR / 'selected_genes_bendpoint.tsv'

cre_selected = pd.read_csv(selected_path, sep='	')
cre_full = pd.read_csv(full_path, sep='	') if full_path.exists() else None

cre_selected['Direction'] = cre_selected['log2FoldChange'].gt(0).map({True: 'Upregulated', False: 'Downregulated'})
cre_selected.loc[cre_selected['log2FoldChange'].eq(0), 'Direction'] = 'Zero'
cre_selected['abs_log2FoldChange'] = cre_selected['log2FoldChange'].abs()

branch_summary = pd.DataFrame({
    'set': ['full significant CRE branch', 'bend-point selected CRE branch', 'selected upregulated', 'selected downregulated', 'selected zero'],
    'n_genes': [
        len(cre_full) if cre_full is not None else pd.NA,
        len(cre_selected),
        int((cre_selected['log2FoldChange'] > 0).sum()),
        int((cre_selected['log2FoldChange'] < 0).sum()),
        int((cre_selected['log2FoldChange'] == 0).sum()),
    ]
})
branch_summary.to_csv(CRE_DIR / 'cre_branch_direction_summary.tsv', sep='	', index=False)
display(branch_summary)


## Anchor-gene companion table — `ipsi_vs_contra_in_cre`

This recreates the FF anchor-gene table for the CRE branch. If a gene-symbol file is not available for this branch, the notebook keeps Ensembl gene IDs as the stable identifiers. The table is meant to support discussion and figure-caption writing, not to replace pathway-level interpretation.


In [ ]:
symbol_path = CRE_DIR / 'selected_genes_bendpoint_gene_symbols.tsv'
anchor_df = cre_selected.copy()
if symbol_path.exists():
    symbol_df = pd.read_csv(symbol_path, sep='	').drop_duplicates('gene_id')
    anchor_df = anchor_df.merge(symbol_df, on='gene_id', how='left')
else:
    anchor_df['symbol'] = anchor_df['gene_id']

anchor_df['symbol'] = anchor_df['symbol'].fillna(anchor_df['gene_id'])
anchor_table = pd.concat([
    anchor_df[anchor_df['log2FoldChange'] > 0].sort_values(['padj', 'abs_log2FoldChange'], ascending=[True, False]).head(10),
    anchor_df[anchor_df['log2FoldChange'] < 0].sort_values(['padj', 'abs_log2FoldChange'], ascending=[True, False]).head(10),
], ignore_index=True).copy()

anchor_table['Gene symbol'] = anchor_table['symbol']
anchor_table['Gene ID'] = anchor_table['gene_id']
anchor_table['Selection note'] = 'top within CRE bend-point core by padj'
anchor_table = anchor_table[['Direction', 'Gene symbol', 'Gene ID', 'log2FoldChange', 'padj', 'baseMean', 'Selection note']]
anchor_table.to_csv(CRE_DIR / 'anchor_genes_up_down.tsv', sep='	', index=False)
display(anchor_table)


## Direction-split GO follow-up — CRE/cKO

As with the old FF notebook, opposite-sign genes should not be mixed when we want a readable biological explanation. This section keeps the CRE GO follow-up split into upregulated and downregulated branches so we can see whether the interpretable signal is symmetric or concentrated in one direction.


In [ ]:
for label, image_name, table_name in [
    ('CRE upregulated', 'gprofiler_terms_and_overlap_up.png', 'gprofiler_enrichment_up.tsv'),
    ('CRE downregulated', 'gprofiler_terms_and_overlap_down.png', 'gprofiler_enrichment_down.tsv'),
]:
    display(Markdown(f'### {label}'))
    image_path = CRE_DIR / image_name
    if image_path.exists():
        display(Image(filename=str(image_path), width=1150))
    else:
        display(Markdown(f'Missing image: `{image_path}`'))
    table_path = CRE_DIR / table_name
    if table_path.exists():
        table = pd.read_csv(table_path, sep='	')
        keep = [col for col in ['source', 'native', 'name', 'p_value', 'term_size', 'query_size', 'intersection_size'] if col in table.columns]
        display(table.loc[:, keep].head(12))
    else:
        display(Markdown(f'Missing table: `{table_path}`'))


## ShinyGO-style views for the CRE branch

This consolidates the CRE-specific ShinyGO-style outputs that are already available locally. The network view is especially useful for deciding whether the CRE branch adds a coherent pathway story or whether the extra genes mostly broaden already-shared injury-response themes.


In [ ]:
for image_path in [
    CRE_DIR / 'gprofiler_top_terms_up.png',
    CRE_DIR / 'gprofiler_source_summary_up.png',
    CRE_SHARED_DIR / 'cre_up_network_view.png',
    CRE_DIR / 'gprofiler_top_terms_down.png',
    CRE_DIR / 'gprofiler_source_summary_down.png',
]:
    if image_path.exists():
        display(Image(filename=str(image_path), width=1100))
    else:
        display(Markdown(f'Missing ShinyGO-style image: `{image_path}`'))


## Pathway themes to investigate

This mirrors the FF pathway-theme table, but for CRE/cKO. The table keeps the paper-facing interpretation honest: broad GO labels should not be overclaimed, but repeated biological-process terms can guide concise discussion around signaling, regulation, localization, phosphorylation/metabolism, and injury-response remodeling.


In [ ]:
cre_theme_table = (
    pd.read_csv(CRE_DIR / 'gprofiler_enrichment.tsv', sep='	')
    .loc[:, ['source', 'native', 'name', 'p_value', 'intersection_size', 'query_size', 'term_size']]
    .query("source in ['GO:BP', 'KEGG', 'REAC']")
    .head(20)
    .copy()
)
cre_theme_table.to_csv(CRE_DIR / 'cre_pathway_theme_table.tsv', sep='	', index=False)
display(cre_theme_table)


## CRE vs FF comparison — shared core and cKO extension

This is the key paper-facing justification for carrying the CRE branch forward. The 870-gene set is useful because it mostly preserves the WT/FF injury-response core while adding a larger cKO-only component. That makes it a supporting extension of the main side-specific story, not a competing replacement for the FF branch.


In [ ]:
ff_selected = pd.read_csv(FF_DIR / 'selected_genes_bendpoint.tsv', sep='	')
cre_selected = pd.read_csv(CRE_DIR / 'selected_genes_bendpoint.tsv', sep='	')

ff_genes = set(ff_selected['gene_id'])
cre_genes = set(cre_selected['gene_id'])
shared = ff_genes & cre_genes
ff_only = ff_genes - cre_genes
cre_only = cre_genes - ff_genes

overlap_summary = pd.DataFrame({
    'category': ['FF/WT selected', 'CRE/cKO selected', 'shared selected genes', 'FF/WT-only selected genes', 'CRE/cKO-only selected genes'],
    'n_genes': [len(ff_genes), len(cre_genes), len(shared), len(ff_only), len(cre_only)]
})
overlap_summary.to_csv(COMPARE_DIR / 'cre_followup_overlap_summary.tsv', sep='	', index=False)
display(overlap_summary)

for image_path in [
    COMPARE_DIR / 'ff_cre_gene_overlap_venn_style.png',
    COMPARE_DIR / 'ff_cre_shared_go_term_scatter.png',
    COMPARE_DIR / 'wt_cko_gene_overlap_venn_style.png',
    COMPARE_DIR / 'wt_cko_shared_go_term_scatter.png',
]:
    if image_path.exists():
        display(Image(filename=str(image_path), width=900))
    else:
        display(Markdown(f'Missing comparison image: `{image_path}`'))


## Paper-ready summary export

Run this cell after checking the displays above. It writes a compact markdown summary that can be reused for the paper, report, or a teammate handoff without reopening the entire notebook.


In [ ]:
summary_md = f"""# CRE/cKO side-specific follow-up summary

- Branch: `ipsi_vs_contra_in_cre`.
- Origin: same 20-sample DRG DESeq2 model used for the FF/WT branch.
- Generation: same ordered adjusted-p-value bend-point procedure used for `ipsi_vs_contra_in_ff`.
- Full significant branch size: {overlap_summary.loc[overlap_summary['category'].eq('CRE/cKO selected'), 'n_genes'].iloc[0]} selected after bend-point; see `bendpoint_summary.tsv` for full branch details.
- Direction split in selected set: {int((cre_selected['log2FoldChange'] > 0).sum())} upregulated and {int((cre_selected['log2FoldChange'] < 0).sum())} downregulated genes.
- FF/CRE overlap: {len(shared)} shared genes, {len(ff_only)} FF/WT-only genes, and {len(cre_only)} CRE/cKO-only genes.
- Interpretation: the CRE branch supports the same side-specific injury-response backbone while expanding the follow-up set under the cKO background.

Useful generated files:
- `cre_branch_direction_summary.tsv`
- `anchor_genes_up_down.tsv`
- `cre_pathway_theme_table.tsv`
- `cre_followup_overlap_summary.tsv`
"""

summary_path = CRE_DIR / 'cre_followup_summary.md'
summary_path.write_text(summary_md)
display(Markdown(summary_md))
print('Wrote:', summary_path)


## Main takeaways

- The CRE/cKO branch is a normal extension of the same side-specific analysis, not a separate hidden workflow.
- The branch starts from the same DESeq2 model family and uses the same bend-point narrowing logic as FF/WT.
- The 870-gene selected set is larger than the 709-gene FF set, but the overlap is substantial enough to support a shared injury-response backbone.
- The CRE-only component is useful for paper discussion because it suggests genotype-background expansion rather than a completely separate story.
- For the paper, FF/WT can remain the cleanest pathway-level showcase, while CRE/cKO should be described as the strongest supporting extension branch.
